# Vision Mark 12 — GPU Training

3-Level Hierarchical Encoder-Decoder with Shared Weights

**Before running:** Go to Runtime → Change runtime type → select **T4 GPU**

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Encoding Table

In [ ]:
# ── Character encoder ──
_id = 1
CHAR_TO_INT = {}

for c in 'abcdefghijklmnopqrstuvwxyz':
    CHAR_TO_INT[c] = _id; _id += 1
for c in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ':
    CHAR_TO_INT[c] = _id; _id += 1
for c in '0123456789':
    CHAR_TO_INT[c] = _id; _id += 1
CHAR_TO_INT[' '] = _id; _id += 1
for c in '!@#$%^&*()-_=+[]{}\\|;:\'",.<>/?`~':
    CHAR_TO_INT[c] = _id; _id += 1

CHAR_TO_INT['<START>'] = _id; _id += 1
CHAR_TO_INT['<INPUT>'] = _id; _id += 1
CHAR_TO_INT['</INPUT>'] = _id; _id += 1

VOCAB_SIZE = _id
START_TOKEN = CHAR_TO_INT['<START>']
INPUT_TOKEN = CHAR_TO_INT['<INPUT>']
END_INPUT_TOKEN = CHAR_TO_INT['</INPUT>']

INT_TO_CHAR = {v: k for k, v in CHAR_TO_INT.items()}
INT_TO_CHAR[0] = ''

def encode_chars(text):
    return [CHAR_TO_INT.get(c, 0) for c in text]

def encode_input(text):
    return [START_TOKEN, INPUT_TOKEN] + encode_chars(text) + [END_INPUT_TOKEN]

def decode_ids(ids):
    return ''.join(INT_TO_CHAR.get(i, '?') for i in ids)

print(f'Vocab size: {VOCAB_SIZE}')
print(f'<START>={START_TOKEN}  <INPUT>={INPUT_TOKEN}  </INPUT>={END_INPUT_TOKEN}')

## 2. Download & Format Training Data

In [ ]:
import urllib.request, re, random, os

DATA_FILE = 'gutenberg_convos.txt'

if not os.path.exists(DATA_FILE):
    books = {
        'Alice in Wonderland': 'https://www.gutenberg.org/cache/epub/11/pg11.txt',
        'Pride and Prejudice': 'https://www.gutenberg.org/cache/epub/1342/pg1342.txt',
        'Sherlock Holmes':     'https://www.gutenberg.org/cache/epub/1661/pg1661.txt',
    }

    all_text = []
    for title, url in books.items():
        data = urllib.request.urlopen(url).read().decode('utf-8', errors='ignore')
        for marker in ['*** START OF THE PROJECT GUTENBERG', '*** START OF THIS PROJECT GUTENBERG']:
            idx = data.find(marker)
            if idx != -1:
                data = data[idx + len(marker):]
                nl = data.find('\n')
                if nl != -1: data = data[nl+1:]
                break
        for marker in ['*** END OF THE PROJECT GUTENBERG', '*** END OF THIS PROJECT GUTENBERG']:
            idx = data.find(marker)
            if idx != -1:
                data = data[:idx]
                break
        all_text.append(data.strip())
        print(f'{title}: {len(data):,} chars')

    raw = '\n\n'.join(all_text)
    for old, new in {'\r':'', '\t':' ', '\u2018':"'", '\u2019':"'",
                      '\u201c':'"', '\u201d':'"', '\u2014':'-',
                      '\u2013':'-', '\ufeff':''}.items():
        raw = raw.replace(old, new)

    raw = re.sub(r'\n+', ' ', raw)
    while '  ' in raw: raw = raw.replace('  ', ' ')

    sentences = re.split(r'(?<=[.!?])\s+', raw)
    sentences = [s.strip() for s in sentences if 10 < len(s.strip()) < 200]
    print(f'Sentences: {len(sentences):,}')

    random.seed(42)
    convos = []
    i = 0
    while i < len(sentences) - 1:
        n_turns = random.randint(1, 3)
        turns = []
        for _ in range(n_turns):
            if i + 1 >= len(sentences): break
            q = sentences[i]
            n_resp = random.randint(1, min(3, len(sentences) - i - 1))
            resp = ' '.join(sentences[i+1:i+1+n_resp])
            turns.append(f'<INPUT>{q}</INPUT>{resp}')
            i += 1 + n_resp
        if turns: convos.append('\n'.join(turns))

    with open(DATA_FILE, 'w') as f:
        f.write('\n\n'.join(convos))
    print(f'Conversations: {len(convos):,} | File: {os.path.getsize(DATA_FILE):,} bytes')
else:
    print(f'Using existing {DATA_FILE} ({os.path.getsize(DATA_FILE):,} bytes)')

## 3. Data Pipeline

In [ ]:
from pathlib import Path
from torch.utils.data import Dataset, DataLoader


def _clean_text(text):
    for old, new in {'\n':' ', '\r':' ', '\t':' ',
                      '\u2018':"'", '\u2019':"'",
                      '\u201c':'"', '\u201d':'"',
                      '\u2014':'-', '\u2013':'-', '\ufeff':''}.items():
        text = text.replace(old, new)
    while '  ' in text: text = text.replace('  ', ' ')
    return text


def parse_conversation_file(path):
    text = Path(path).read_text(encoding='utf-8', errors='ignore')
    chunks = re.split(r'\n\s*\n', text.strip())
    all_convos = []
    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk: continue
        pattern = re.compile(r'<INPUT>(.*?)</INPUT>(.*?)(?=<INPUT>|$)', re.DOTALL)
        matches = pattern.findall(chunk)
        turns = []
        for user_msg, response in matches:
            user_msg = user_msg.strip()
            response = _clean_text(response).strip()
            if user_msg and response:
                turns.append((user_msg, response))
        if turns: all_convos.append(turns)
    return all_convos


def encode_conversation(turns):
    encoded_pairs = []
    running_prefix = [START_TOKEN]
    for user_msg, response in turns:
        user_ids = encode_chars(user_msg)
        response_ids = encode_chars(response)
        turn_prefix = running_prefix + [INPUT_TOKEN] + user_ids + [END_INPUT_TOKEN]
        if response_ids:
            encoded_pairs.append((list(turn_prefix), response_ids))
        running_prefix = turn_prefix + response_ids
    return encoded_pairs


class ConversationDataset(Dataset):
    def __init__(self, encoded_pairs, max_context=512):
        self.pairs = encoded_pairs
        self.max_context = max_context
        self.index = []
        for i, (prefix, response) in enumerate(encoded_pairs):
            for j in range(len(response)):
                self.index.append((i, j))

    def __len__(self): return len(self.index)

    def __getitem__(self, idx):
        pair_idx, resp_pos = self.index[idx]
        prefix, response = self.pairs[pair_idx]
        input_ids = prefix + response[:resp_pos]
        if len(input_ids) > self.max_context:
            input_ids = input_ids[-self.max_context:]
        target = response[resp_pos]
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(target, dtype=torch.long)


def collate_variable_length(batch):
    inputs, targets = zip(*batch)
    max_len = max(x.shape[0] for x in inputs)
    padded = torch.zeros(len(inputs), max_len, dtype=torch.long)
    for i, x in enumerate(inputs):
        padded[i, :x.shape[0]] = x
    return padded, torch.stack(targets)


# Build datasets
all_convos = parse_conversation_file(DATA_FILE)
random.shuffle(all_convos)
split_idx = int(len(all_convos) * 0.9)

train_pairs, val_pairs = [], []
for c in all_convos[:split_idx]: train_pairs.extend(encode_conversation(c))
for c in all_convos[split_idx:]: val_pairs.extend(encode_conversation(c))

train_ds = ConversationDataset(train_pairs)
val_ds = ConversationDataset(val_pairs)

total_turns = sum(len(c) for c in all_convos)
total_resp = sum(len(r) for _, r in train_pairs + val_pairs)
print(f'Conversations: {len(all_convos):,} ({total_turns:,} turns, {total_resp:,} response chars)')
print(f'Train: {len(train_ds):,} examples | Val: {len(val_ds):,} examples')

## 4. Model Architecture

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class Scanner(nn.Module):
    """P scanner-detail pairs per fan_in. Encode + decode, shared weights."""

    def __init__(self, input_dim, max_fan_in=20, pairs_per_fan_in=3,
                 space_token=None, space_init=-100.0):
        super().__init__()
        self.input_dim = input_dim
        self.max_fan_in = max_fan_in
        self.pairs = pairs_per_fan_in
        self.space_token = space_token
        self.values_per_unit = max_fan_in * pairs_per_fan_in

        self.scan_weights = nn.ParameterList()
        for k in range(1, max_fan_in + 1):
            w = nn.Parameter(torch.empty(pairs_per_fan_in, k, input_dim))
            nn.init.kaiming_uniform_(w.view(pairs_per_fan_in, -1))
            self.scan_weights.append(w)

        self.detail_weights = nn.ParameterList()
        for k in range(1, max_fan_in + 1):
            w = nn.Parameter(torch.empty(pairs_per_fan_in, k, input_dim))
            nn.init.kaiming_uniform_(w.view(pairs_per_fan_in, -1))
            self.detail_weights.append(w)

        if space_token is not None and input_dim > space_token:
            with torch.no_grad():
                for w in self.scan_weights:
                    w[:, :, space_token] = space_init
                for w in self.detail_weights:
                    w[:, :, space_token] = space_init

    def forward(self, x):
        batch, seq_len, _ = x.shape
        device = x.device
        K = self.max_fan_in
        P = self.pairs

        scan_alive = torch.zeros(batch, seq_len, K, dtype=torch.bool, device=device)
        group_outputs = torch.zeros(batch, seq_len, K * P, device=device)

        for k_idx, k in enumerate(range(1, K + 1)):
            if k > seq_len: break
            windows = x.unfold(1, k, 1).permute(0, 1, 3, 2)
            sw = self.scan_weights[k_idx]
            scan_acts = F.relu(torch.einsum('bpki,ski->bps', windows, sw))
            scan_alive[:, :seq_len-k+1, k_idx] = (scan_acts > 0).any(dim=-1)
            dw = self.detail_weights[k_idx]
            detail_acts = F.relu(torch.einsum('bpki,ski->bps', windows, dw))
            gated = scan_acts * detail_acts
            group_outputs[:, :seq_len-k+1, k_idx*P:k_idx*P+P] = gated

        priority = torch.arange(K, device=device).view(1, 1, -1).expand(batch, seq_len, -1)
        priority_masked = torch.where(scan_alive, priority, torch.tensor(-1, device=device))
        best_k_idx = priority_masked.argmax(dim=-1)
        any_alive = scan_alive.any(dim=-1)
        best_fan_in = (best_k_idx + 1) * any_alive.long()

        survivor_positions = torch.zeros(batch, seq_len, dtype=torch.long, device=device)
        survivor_fan_ins = torch.zeros(batch, seq_len, dtype=torch.long, device=device)
        survivor_counts = torch.zeros(batch, dtype=torch.long, device=device)

        for b in range(batch):
            fi_b = best_fan_in[b]
            pos = 0
            count = 0
            while pos < seq_len and count < seq_len:
                fi = fi_b[pos].item()
                if fi > 0:
                    survivor_positions[b, count] = pos
                    survivor_fan_ins[b, count] = fi
                    count += 1
                    pos += fi
                else:
                    pos += 1
            survivor_counts[b] = count

        max_surv = max(survivor_counts.max().item(), 1)
        surv_pos = survivor_positions[:, :max_surv].clamp(0, seq_len - 1)
        surv_fi = survivor_fan_ins[:, :max_surv]

        pos_idx = surv_pos.unsqueeze(-1).expand(-1, -1, K * P)
        values_at_surv = group_outputs.gather(1, pos_idx)

        group_fan_ins = torch.arange(1, K+1, device=device).repeat_interleave(P).view(1, 1, K*P)
        length_mask = (group_fan_ins <= surv_fi.unsqueeze(-1)).float()
        values_at_surv = values_at_surv * length_mask

        surv_mask = torch.arange(max_surv, device=device).unsqueeze(0) < survivor_counts.unsqueeze(1)
        values_at_surv = values_at_surv * surv_mask.unsqueeze(-1).float()

        return values_at_surv

    def decode(self, values):
        K = self.max_fan_in
        P = self.pairs
        leading_shape = values.shape[:-1]
        device = values.device
        expanded = torch.zeros(*leading_shape, K, self.input_dim, device=device)
        for k_idx in range(K):
            k = k_idx + 1
            dw = self.detail_weights[k_idx]
            for p in range(P):
                v = values[..., k_idx * P + p]
                w = dw[p]
                contrib = v.unsqueeze(-1).unsqueeze(-1) * w
                expanded[..., :k, :] = expanded[..., :k, :] + contrib
        return expanded


class RMSNorm1d(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x.float().pow(2).mean(1, keepdim=True).add(self.eps).rsqrt()
        return (x.float() * norm).type_as(x) * self.weight.view(1, -1, 1)


class VM7Model(nn.Module):
    def __init__(self, vocab_size=99, pairs=3, wK=20, pK=20, iK=20,
                 dense_w=1000, space_token=63, space_init=-100.0):
        super().__init__()
        self.vocab_size = vocab_size

        self.word_scanner = Scanner(
            input_dim=vocab_size, max_fan_in=wK, pairs_per_fan_in=pairs,
            space_token=space_token, space_init=space_init)
        word_dim = self.word_scanner.values_per_unit

        self.phrase_scanner = Scanner(
            input_dim=word_dim, max_fan_in=pK, pairs_per_fan_in=pairs)
        phrase_dim = self.phrase_scanner.values_per_unit

        self.idea_scanner = Scanner(
            input_dim=phrase_dim, max_fan_in=iK, pairs_per_fan_in=pairs)
        idea_dim = self.idea_scanner.values_per_unit

        self.dense1 = nn.Conv1d(idea_dim, dense_w, 1, bias=False)
        self.dense_norm2 = RMSNorm1d(dense_w)
        self.dense2 = nn.Conv1d(dense_w, dense_w, 1, bias=False)
        self.dense_norm3 = RMSNorm1d(dense_w)
        self.dense3 = nn.Conv1d(dense_w, dense_w, 1, bias=False)
        self.dense_norm4 = RMSNorm1d(dense_w)
        self.dense4 = nn.Conv1d(dense_w, dense_w, 1, bias=False)

        self.output = nn.Linear(dense_w, vocab_size, bias=False)
        self.project_to_idea = nn.Linear(dense_w, idea_dim)

        self._info = dict(vocab_size=vocab_size, pairs=pairs,
                          wK=wK, pK=pK, iK=iK, dense_w=dense_w,
                          word_dim=word_dim, phrase_dim=phrase_dim,
                          idea_dim=idea_dim)

    def _encode_and_process(self, x):
        x_oh = F.one_hot(x, num_classes=self.vocab_size).float()
        word_values = self.word_scanner(x_oh)
        phrase_values = self.phrase_scanner(word_values)
        idea_values = self.idea_scanner(phrase_values)
        h = idea_values.permute(0, 2, 1)
        h = F.relu(self.dense1(h))
        h = F.relu(self.dense2(self.dense_norm2(h)))
        h = F.relu(self.dense3(self.dense_norm3(h)))
        h = F.relu(self.dense4(self.dense_norm4(h)))
        return h

    def forward(self, x):
        h = self._encode_and_process(x)
        return self.output(h[:, :, -1])

    def decode_idea(self, x):
        h = self._encode_and_process(x)
        idea_repr = self.project_to_idea(h[:, :, -1])
        phrase_values = self.idea_scanner.decode(idea_repr)
        word_values = self.phrase_scanner.decode(phrase_values)
        return self.word_scanner.decode(word_values)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def layer_info(self):
        i = self._info
        ws = sum(p.numel() for p in self.word_scanner.parameters())
        ps = sum(p.numel() for p in self.phrase_scanner.parameters())
        iss = sum(p.numel() for p in self.idea_scanner.parameters())
        dp = sum(sum(p.numel() for p in l.parameters()) for l in
                 [self.dense1,self.dense2,self.dense3,self.dense4,
                  self.dense_norm2,self.dense_norm3,self.dense_norm4])
        pp = sum(p.numel() for p in self.project_to_idea.parameters())
        op = sum(p.numel() for p in self.output.parameters())
        lines = [
            f'=== Vision Mark 12 Architecture ===',
            f'Vocab: {i["vocab_size"]} | {i["pairs"]} pairs per fan_in',
            f'',
            f'--- 3 SCANNERS (encode <-> decode, shared weights) ---',
            f'L1 Word:   chars({i["vocab_size"]}) -> {i["word_dim"]} nums/word ({i["pairs"]}x{i["wK"]}) -- {ws:,} params',
            f'L2 Phrase: words({i["word_dim"]}) -> {i["phrase_dim"]} nums/phrase ({i["pairs"]}x{i["pK"]}) -- {ps:,} params',
            f'L3 Idea:   phrases({i["phrase_dim"]}) -> {i["idea_dim"]} nums/idea ({i["pairs"]}x{i["iK"]}) -- {iss:,} params',
            f'',
            f'--- PROCESSING ---',
            f'Dense: {i["idea_dim"]}->{i["dense_w"]}->{i["dense_w"]}->{i["dense_w"]}->{i["dense_w"]} -- {dp:,} params',
            f'Project: {i["dense_w"]}->{i["idea_dim"]} -- {pp:,} params',
            f'',
            f'Next-char head: Linear({i["dense_w"]}, {i["vocab_size"]}) -- {op:,} params',
            f'Total parameters: {self.count_parameters():,}',
        ]
        return '\n'.join(lines)


model = VM7Model(vocab_size=VOCAB_SIZE)
print(model.layer_info())

## 5. Training Loop

In [ ]:
import time

# ── Config ──
LR = 3e-4
BATCH_SIZE = 32
MAX_STEPS = 50_000
GRAD_CLIP = 1.0
LOG_EVERY = 100
VAL_EVERY = 500
SAVE_EVERY = 2000

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = model.to(device)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate_variable_length,
                          pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, collate_fn=collate_variable_length,
                        pin_memory=True, drop_last=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

demo_prompts = [
    'What is the capital of France?',
    'Tell me about dogs',
    'How does the sun work?',
    'What is 2+2?',
]


@torch.no_grad()
def evaluate(max_batches=50):
    model.eval()
    total_loss = total_correct = total_count = num_b = 0
    for i, (x, y) in enumerate(val_loader):
        if i >= max_batches: break
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += F.cross_entropy(logits, y).item()
        total_correct += (logits.argmax(-1) == y).sum().item()
        total_count += y.size(0)
        num_b += 1
    model.train()
    return total_loss / max(num_b, 1), total_correct / max(total_count, 1)


@torch.no_grad()
def demo(text, n_chars=40):
    model.eval()
    ids = encode_input(text)
    gen = []
    for _ in range(n_chars):
        x = torch.tensor([ids], dtype=torch.long, device=device)
        pred = model(x)[0].argmax().item()
        gen.append(pred)
        ids.append(pred)
    model.train()
    return decode_ids(gen)


def save_ckpt(path, step, best_val):
    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                'step': step, 'best_val_loss': best_val}, path)


# ── Train ──
model.train()
step = 0
best_val_loss = float('inf')
running_loss = running_correct = running_count = 0
t0 = time.time()

print(f'\nTraining for {MAX_STEPS:,} steps, batch={BATCH_SIZE}, lr={LR}')
print(f'Train: {len(train_ds):,} examples | Val: {len(val_ds):,} examples\n')

while step < MAX_STEPS:
    for x, y in train_loader:
        if step >= MAX_STEPS: break
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        optimizer.zero_grad()

        running_loss += loss.item()
        running_correct += (logits.argmax(-1) == y).sum().item()
        running_count += y.size(0)
        step += 1

        if step % LOG_EVERY == 0:
            avg = running_loss / LOG_EVERY
            acc = running_correct / max(running_count, 1)
            elapsed = time.time() - t0
            print(f'Step {step:>6d} | loss: {avg:.4f} | acc: {acc:.3f} | {elapsed:.0f}s')
            running_loss = running_correct = running_count = 0

        if step % VAL_EVERY == 0:
            vl, va = evaluate()
            print(f'  >>> val_loss: {vl:.4f} | val_acc: {va:.3f}')
            if vl < best_val_loss:
                best_val_loss = vl
                save_ckpt('best.pt', step, best_val_loss)
                print(f'  >>> New best! Saved best.pt')
            for p in demo_prompts:
                print(f'  >>> <INPUT>"{p}" -> "{demo(p)}"')

        if step % SAVE_EVERY == 0:
            save_ckpt('latest.pt', step, best_val_loss)
            print(f'  >>> Saved latest.pt at step {step}')

save_ckpt('final.pt', step, best_val_loss)
print(f'\nDone! {step:,} steps in {time.time()-t0:.0f}s')

## 6. Test the Model

In [ ]:
# Load best checkpoint
ckpt = torch.load('best.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'])
print(f'Loaded best.pt from step {ckpt["step"]}, val_loss={ckpt["best_val_loss"]:.4f}')

test_prompts = [
    'Hello, how are you?',
    'What is the meaning of life?',
    'Tell me about Alice',
    'Who is Sherlock Holmes?',
    'Describe a beautiful day',
]

for p in test_prompts:
    result = demo(p, n_chars=80)
    print(f'\n<INPUT>"{p}"')
    print(f'  -> "{result}"')

## 7. Download Checkpoint

Run this to download the trained model back to your machine.

In [ ]:
from google.colab import files
files.download('best.pt')